# Temperature

In [6]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/tas/CMCC-CM2-SR5_tas_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/tas/CMCC-CM2-SR5_tas_present_19812010_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(1981, 2010))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["tas"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["tas"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["tas"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean) - keep in Kelvin for percentage calculation
    baseline_climatology_K = base_annual_means.mean()
    
    # Calculate temperature changes for each year in 1981-2010 relative to baseline
    temperature_changes = present_annual_means - baseline_climatology_K
    
    # Convert absolute changes to Celsius for reporting
    temperature_changes_celsius = temperature_changes
    
    # Mean and standard deviation of temperature changes in Celsius
    mean_change_celsius = temperature_changes_celsius.mean().item()
    std_change_celsius = temperature_changes_celsius.std(ddof=1).item()
    
    # Calculate percentage changes using Kelvin baseline
    pct_change_mean = (mean_change_celsius / baseline_climatology_K.item()) * 100
    pct_change_std = (std_change_celsius / baseline_climatology_K.item()) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_C": mean_change_celsius,
        "Std_Change_C": std_change_celsius,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Temperature Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(°C)':>12}{'(K basis)':>10}{'(°C)':>12}{'(K basis)':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_C']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_C']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average temperature change over 30 years (1981-2010) in °C")
print(f"Mean %: Percentage of mean temperature change relative to baseline (using Kelvin)")
print(f"Std Change: Standard deviation of the 30 annual temperature changes in °C")
print(f"Std %: Percentage of standard deviation relative to baseline (using Kelvin)")

Temperature Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                            (°C) (K basis)        (°C) (K basis)
----------------------------------------------------------------------
Global                     0.226      0.08       0.281      0.10
Tropics                    0.190      0.06       0.290      0.10
Subtropics_N               0.276      0.09       0.359      0.12
Subtropics_S               0.150      0.05       0.204      0.07
Mid_Latitudes_N            0.413      0.15       0.588      0.21
Mid_Latitudes_S            0.147      0.05       0.190      0.07

Mean Change: Average temperature change over 30 years (1981-2010) in °C
Mean %: Percentage of mean temperature change relative to baseline (using Kelvin)
Std Change: Standard deviation of the 30 annual temperature changes in °C
Std %: Percentage of standard deviation relative to baseline (using Kelvin)


Future

In [29]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/tas/CMCC-CM2-SR5_tas_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/tas/CMCC-CM2-SR5_tas_future585_20212050_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2021, 2050))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["tas"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["tas"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for future period (2021-2050) 
    present_annual_means = (present_reg["tas"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean) - keep in Kelvin for percentage calculation
    baseline_climatology_K = base_annual_means.mean()
    
    # Calculate temperature changes for each year in 2021-2050 relative to baseline
    temperature_changes = present_annual_means - baseline_climatology_K
    
    # Absolute changes remain in °C (the difference is the same in K or °C)
    temperature_changes_celsius = temperature_changes
    
    # Mean and standard deviation of temperature changes in Celsius
    mean_change_celsius = temperature_changes_celsius.mean().item()
    std_change_celsius = temperature_changes_celsius.std(ddof=1).item()
    
    # Calculate percentage changes using Kelvin baseline
    pct_change_mean = (mean_change_celsius / baseline_climatology_K.item()) * 100
    pct_change_std = (std_change_celsius / baseline_climatology_K.item()) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_C": mean_change_celsius,
        "Std_Change_C": std_change_celsius,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Temperature Changes: 2021-2050 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(°C)':>12}{'(K basis)':>10}{'(°C)':>12}{'(K basis)':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_C']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_C']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average temperature change over 30 years (2021-2050) in °C")
print(f"Mean %: Percentage of mean temperature change relative to baseline (using Kelvin)")
print(f"Std Change: Standard deviation of the 30 annual temperature changes in °C")
print(f"Std %: Percentage of standard deviation relative to baseline (using Kelvin)")

Temperature Changes: 2021-2050 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                            (°C) (K basis)        (°C) (K basis)
----------------------------------------------------------------------
Global                     1.354      0.46       0.454      0.16
Tropics                    1.223      0.41       0.468      0.16
Subtropics_N               1.592      0.54       0.543      0.19
Subtropics_S               1.117      0.38       0.347      0.12
Mid_Latitudes_N            2.056      0.73       0.799      0.28
Mid_Latitudes_S            1.003      0.36       0.270      0.10

Mean Change: Average temperature change over 30 years (2021-2050) in °C
Mean %: Percentage of mean temperature change relative to baseline (using Kelvin)
Std Change: Standard deviation of the 30 annual temperature changes in °C
Std %: Percentage of standard deviation relative to baseline (using Kelvin)


# Precipitable Water

In [49]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_present_19812010_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(1981, 2010))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["prw"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate precipitable water changes for each year in 1981-2010 relative to baseline
    prw_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m2 to mm (1 kg/m2 = 1 mm)
    baseline_mm = baseline_climatology.item()
    prw_changes_mm = prw_changes
    
    # Mean and standard deviation of precipitable water changes in mm
    mean_change_mm = prw_changes_mm.mean().item()
    std_change_mm = prw_changes_mm.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mm / baseline_mm) * 100
    pct_change_std = (std_change_mm / baseline_mm) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mm": mean_change_mm,
        "Std_Change_mm": std_change_mm,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitable Water Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm)':>12}{'':>10}{'(mm)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mm']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mm']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitable water change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean precipitable water change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitable water changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitable Water Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                            (mm)                  (mm)          
----------------------------------------------------------------------
Global                     0.246      0.91       0.669      2.47
Tropics                    0.246      0.65       1.203      3.17
Subtropics_N               0.463      2.02       0.565      2.47
Subtropics_S               0.160      0.67       0.268      1.12
Mid_Latitudes_N            0.385      2.66       0.413      2.85
Mid_Latitudes_S            0.032      0.21       0.145      0.96

Mean Change: Average precipitable water change over 30 years (1981-2010)
Mean %: Percentage of mean precipitable water change relative to baseline
Std Change: Standard deviation of the 30 annual precipitable water changes
Std %: Percentage of standard deviation relative to baseline


Future

In [83]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_future585_20712100_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["prw"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate precipitable water changes for each year in 1981-2010 relative to baseline
    prw_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m2 to mm (1 kg/m2 = 1 mm)
    baseline_mm = baseline_climatology.item()
    prw_changes_mm = prw_changes
    
    # Mean and standard deviation of precipitable water changes in mm
    mean_change_mm = prw_changes_mm.mean().item()
    std_change_mm = prw_changes_mm.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mm / baseline_mm) * 100
    pct_change_std = (std_change_mm / baseline_mm) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mm": mean_change_mm,
        "Std_Change_mm": std_change_mm,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitable Water Changes: 2021-2050 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm)':>12}{'':>10}{'(mm)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mm']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mm']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitable water change over 30 years (2021-2050)")
print(f"Mean %: Percentage of mean precipitable water change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitable water changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitable Water Changes: 2021-2050 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                            (mm)                  (mm)          
----------------------------------------------------------------------
Global                     7.354     27.11       1.187      4.38
Tropics                   10.477     27.63       1.772      4.67
Subtropics_N               7.527     32.91       1.064      4.65
Subtropics_S               4.444     18.63       0.795      3.34
Mid_Latitudes_N            5.393     37.27       0.926      6.40
Mid_Latitudes_S            2.496     16.52       0.472      3.12

Mean Change: Average precipitable water change over 30 years (2021-2050)
Mean %: Percentage of mean precipitable water change relative to baseline
Std Change: Standard deviation of the 30 annual precipitable water changes
Std %: Percentage of standard deviation relative to baseline


CMCC

In [70]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/prw/CMCC-CM2-SR5_prw_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/prw/CMCC-CM2-SR5_prw_future585_20212050_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2021, 2050))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["prw"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["prw"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate precipitable water changes for each year in 1981-2010 relative to baseline
    prw_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m2 to mm (1 kg/m2 = 1 mm)
    baseline_mm = baseline_climatology.item()
    prw_changes_mm = prw_changes
    
    # Mean and standard deviation of precipitable water changes in mm
    mean_change_mm = prw_changes_mm.mean().item()
    std_change_mm = prw_changes_mm.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mm / baseline_mm) * 100
    pct_change_std = (std_change_mm / baseline_mm) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mm": mean_change_mm,
        "Std_Change_mm": std_change_mm,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitable Water Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm)':>12}{'':>10}{'(mm)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mm']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mm']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitable water change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean precipitable water change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitable water changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitable Water Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                            (mm)                  (mm)          
----------------------------------------------------------------------
Global                     3.064     10.67       1.120      3.90
Tropics                    4.166     10.35       1.636      4.06
Subtropics_N               3.108     12.74       1.185      4.86
Subtropics_S               2.192      8.67       0.772      3.06
Mid_Latitudes_N            2.326     14.85       0.891      5.68
Mid_Latitudes_S            1.316      8.64       0.415      2.72

Mean Change: Average precipitable water change over 30 years (1981-2010)
Mean %: Percentage of mean precipitable water change relative to baseline
Std Change: Standard deviation of the 30 annual precipitable water changes
Std %: Percentage of standard deviation relative to baseline


# Precipitation

In [121]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_future585_20712100_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["pr"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["pr"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["pr"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean) - FIXED: use mean() not sum()
    baseline_climatology = base_annual_means.mean()
    
    # Calculate precipitation changes for each year in 1981-2010 relative to baseline
    precipitation_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m²/s to mm/day (multiply by 86400)
    baseline_mmday = baseline_climatology.item() * 86400
    precipitation_changes_mmday = precipitation_changes * 86400
    
    # Mean and standard deviation of precipitation changes in mm/day - FIXED: use mean() not sum()
    mean_change_mmday = precipitation_changes_mmday.mean().item()
    std_change_mmday = precipitation_changes_mmday.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitation Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitation change over 30 years (2021-2050)")
print(f"Mean %: Percentage of mean precipitation change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitation changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitation Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
----------------------------------------------------------------------
Global                     0.105      3.45       0.027      0.90
Tropics                    0.154      4.17       0.048      1.30
Subtropics_N               0.108      5.25       0.072      3.50
Subtropics_S              -0.120     -5.09       0.073      3.10
Mid_Latitudes_N            0.171      7.13       0.053      2.21
Mid_Latitudes_S            0.038      1.29       0.037      1.27

Mean Change: Average precipitation change over 30 years (2021-2050)
Mean %: Percentage of mean precipitation change relative to baseline
Std Change: Standard deviation of the 30 annual precipitation changes
Std %: Percentage of standard deviation relative to baseline


CMCC

In [96]:
# Open the NetCDF files
ds1 = xr.open_dataset("/Volumes/Extreme SSD/CMIP6/CMCC-CM2-SR5/pr/historical/pr_day_CMCC-CM2-SR5_historical_r1i1p1f1_gn_19500101-19741231.nc")
ds2 = xr.open_dataset("/Volumes/Extreme SSD/CMIP6/CMCC-CM2-SR5/pr/historical/pr_day_CMCC-CM2-SR5_historical_r1i1p1f1_gn_19750101-19991231.nc")
ds3 = xr.open_dataset("/Volumes/Extreme SSD/CMIP6/CMCC-CM2-SR5/pr/historical/pr_day_CMCC-CM2-SR5_historical_r1i1p1f1_gn_20000101-20141231.nc")
#ds4 = xr.open_dataset("/Volumes/Extreme SSD/CMIP6/NorESM2-MM/pr/historical/pr_day_NorESM2-MM_historical_r1i1p1f1_gn_20100101-20141231.nc")

In [97]:
# Select the time range from each dataset
ds1_selected = ds1.sel(time=slice('1971-01-01', '1974-12-31'))
ds2_selected = ds2.sel(time=slice('1975-01-01', '1999-12-31'))
ds3_selected = ds3.sel(time=slice('2000-01-01', '2000-12-31'))
#ds4_selected = ds4.sel(time=slice('2010-01-01', '2010-11-30'))

In [98]:
# Concatenate the datasets along the time dimension
combined_ds = xr.concat([ds1_selected, ds2_selected, ds3_selected], dim='time')

In [99]:
# Save the combined dataset to a a new NetCDF file
combined_ds.to_netcdf("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/pr/CMCC-CM2-SR5_pr_base_19712000_annual1.nc")

In [101]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/pr/CMCC-CM2-SR5_pr_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-CM2-SR5/pr/CMCC-CM2-SR5_pr_present_19812010_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(1981, 2010))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["pr"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["pr"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["pr"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean) - FIXED: use mean() not sum()
    baseline_climatology = base_annual_means.mean()
    
    # Calculate precipitation changes for each year in 1981-2010 relative to baseline
    precipitation_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m²/s to mm/day (multiply by 86400)
    baseline_mmday = baseline_climatology.item() * 86400
    precipitation_changes_mmday = precipitation_changes * 86400
    
    # Mean and standard deviation of precipitation changes in mm/day - FIXED: use mean() not sum()
    mean_change_mmday = precipitation_changes_mmday.mean().item()
    std_change_mmday = precipitation_changes_mmday.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Precipitation Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average precipitation change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean precipitation change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual precipitation changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Precipitation Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
----------------------------------------------------------------------
Global                     0.006      0.19       0.020      0.59
Tropics                    0.010      0.23       0.034      0.82
Subtropics_N              -0.011     -0.50       0.061      2.65
Subtropics_S              -0.009     -0.37       0.074      3.06
Mid_Latitudes_N            0.014      0.55       0.043      1.65
Mid_Latitudes_S            0.009      0.31       0.035      1.18

Mean Change: Average precipitation change over 30 years (1981-2010)
Mean %: Percentage of mean precipitation change relative to baseline
Std Change: Standard deviation of the 30 annual precipitation changes
Std %: Percentage of standard deviation relative to baseline


# Evaporation

In [147]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/hfls/NorESM2-MM_hfls_future126_20712100_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    # FIXED: Use "hfls" instead of "pr"
    weights_2d = weights.broadcast_like(base_reg["hfls"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    # FIXED: Use "hfls" instead of "pr"
    base_annual_means = (base_reg["hfls"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    # FIXED: Use "hfls" instead of "pr"
    present_annual_means = (present_reg["hfls"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate latent heat flux changes for each year in 1981-2010 relative to baseline
    hfls_changes = present_annual_means - baseline_climatology
    
    # Convert from W/m² to mm/day equivalent evaporation
    # FIXED: Multiply by 86400 first, then divide by latent heat of vaporization
    baseline_mmday = baseline_climatology.item() * 86400 / (2.45 * 1e6)  # W/m² to mm/day
    evaporation_changes_mmday = hfls_changes * 86400 / (2.45 * 1e6)  # W/m² to mm/day
    
    # Mean and standard deviation of evaporation changes in mm/day
    mean_change_mmday = evaporation_changes_mmday.mean().item()
    std_change_mmday = evaporation_changes_mmday.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Latent Heat Flux (Evaporation) Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average evaporation change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean evaporation change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual evaporation changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Latent Heat Flux (Evaporation) Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
----------------------------------------------------------------------
Global                     0.085      2.60       0.017      0.52
Tropics                    0.108      2.61       0.033      0.81
Subtropics_N               0.187      6.37       0.031      1.07
Subtropics_S               0.066      1.82       0.039      1.06
Mid_Latitudes_N            0.064      3.58       0.016      0.91
Mid_Latitudes_S           -0.005     -0.22       0.020      0.85

Mean Change: Average evaporation change over 30 years (1981-2010)
Mean %: Percentage of mean evaporation change relative to baseline
Std Change: Standard deviation of the 30 annual evaporation changes
Std %: Percentage of standard deviation relative to baseline


In [138]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-LM/hfls/NorESM2-LM_hfls_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-LM/hfls/NorESM2-LM_hfls_future126_20712100_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    # FIXED: Use "hfls" instead of "pr"
    weights_2d = weights.broadcast_like(base_reg["hfls"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    # FIXED: Use "hfls" instead of "pr"
    base_annual_means = (base_reg["hfls"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    # FIXED: Use "hfls" instead of "pr"
    present_annual_means = (present_reg["hfls"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate latent heat flux changes for each year in 1981-2010 relative to baseline
    hfls_changes = present_annual_means - baseline_climatology
    
    # Convert from W/m² to mm/day equivalent evaporation
    # FIXED: Multiply by 86400 first, then divide by latent heat of vaporization
    baseline_mmday = baseline_climatology.item() * 86400 / (2.45 * 1e6)  # W/m² to mm/day
    evaporation_changes_mmday = hfls_changes * 86400 / (2.45 * 1e6)  # W/m² to mm/day
    
    # Mean and standard deviation of evaporation changes in mm/day
    mean_change_mmday = evaporation_changes_mmday.mean().item()
    std_change_mmday = evaporation_changes_mmday.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std
    })

# --- Print results ---
print("Latent Heat Flux (Evaporation) Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 70)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.3f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.3f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average evaporation change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean evaporation change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual evaporation changes")
print(f"Std %: Percentage of standard deviation relative to baseline")

Latent Heat Flux (Evaporation) Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
----------------------------------------------------------------------
Global                     0.080      2.45       0.021      0.66
Tropics                    0.097      2.33       0.043      1.04
Subtropics_N               0.226      7.68       0.034      1.15
Subtropics_S               0.051      1.38       0.036      0.97
Mid_Latitudes_N            0.051      2.76       0.019      1.05
Mid_Latitudes_S            0.000      0.01       0.017      0.72

Mean Change: Average evaporation change over 30 years (1981-2010)
Mean %: Percentage of mean evaporation change relative to baseline
Std Change: Standard deviation of the 30 annual evaporation changes
Std %: Percentage of standard deviation relative to baseline


# Soil Moisture

In [158]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrsos/CMCC-ESM2_mrsos_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrsos/CMCC-ESM2_mrsos_present_19812010_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(1981, 2010))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["mrsos"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["mrsos"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["mrsos"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate soil moisture changes for each year in 1981-2010 relative to baseline
    mrsos_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m² to m³/m³ (volumetric water content)
    # Assumptions: 
    # - Soil layer depth: 0.1 m (10 cm) - typical for mrsos
    # - Water density: 1000 kg/m³
    # Formula: (kg/m²) / (water_density × soil_depth) = m³/m³
    
    soil_depth = 0.1  # meters (10 cm soil layer)
    water_density = 1000  # kg/m³
    
    # Convert to volumetric water content (m³/m³)
    baseline_m3m3 = baseline_climatology.item() / (water_density * soil_depth)
    soil_moisture_changes_m3m3 = mrsos_changes / (water_density * soil_depth)
    
    # Mean and standard deviation of soil moisture changes in m³/m³
    mean_change_m3m3 = soil_moisture_changes_m3m3.mean().item()
    std_change_m3m3 = soil_moisture_changes_m3m3.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_m3m3 / baseline_m3m3) * 100
    pct_change_std = (std_change_m3m3 / baseline_m3m3) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_m3m3": mean_change_m3m3,
        "Std_Change_m3m3": std_change_m3m3,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std,
        "Baseline_m3m3": baseline_m3m3
    })

# --- Print results ---
print("Soil Moisture Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(m³/m³)':>12}{'':>10}{'(m³/m³)':>12}{'':>10}")
print("-" * 80)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_m3m3']:>12.6f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_m3m3']:>12.6f}{res['Pct_Change_Std']:>10.2f}")

print(f"Mean %: Percentage of mean soil moisture change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual soil moisture changes")
print(f"Std %: Percentage of standard deviation relative to baseline")
print(f"Note: Assumes 10 cm soil depth and water density of 1000 kg/m³")
print(f"Conversion formula: (kg/m²) / (1000 kg/m³ × 0.1 m) = m³/m³")

Soil Moisture Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                         (m³/m³)               (m³/m³)          
--------------------------------------------------------------------------------
Global                 -0.000136     -0.20    0.000432      0.64
Tropics                -0.000026     -0.04    0.001107      1.78
Subtropics_N            0.000209      0.28    0.001127      1.52
Subtropics_S           -0.000093     -0.24    0.001312      3.40
Mid_Latitudes_N        -0.000768     -0.50    0.001984      1.29
Mid_Latitudes_S        -0.000027     -0.29    0.000175      1.88
Mean %: Percentage of mean soil moisture change relative to baseline
Std Change: Standard deviation of the 30 annual soil moisture changes
Std %: Percentage of standard deviation relative to baseline
Note: Assumes 10 cm soil depth and water density of 1000 kg/m³
Conversion formula: (kg/m²) / (1000 kg/m³ × 0.1 m) = m³/m³


In [178]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_future585_20712100_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["mrsos"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["mrsos"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["mrsos"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate soil moisture changes for each year in 1981-2010 relative to baseline
    mrsos_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m² to m³/m³ (volumetric water content)
    # Assumptions: 
    # - Soil layer depth: 0.1 m (10 cm) - typical for mrsos
    # - Water density: 1000 kg/m³
    # Formula: (kg/m²) / (water_density × soil_depth) = m³/m³
    
    soil_depth = 0.1  # meters (10 cm soil layer)
    water_density = 1000  # kg/m³
    
    # Convert to volumetric water content (m³/m³)
    baseline_m3m3 = baseline_climatology.item() / (water_density * soil_depth)
    soil_moisture_changes_m3m3 = mrsos_changes / (water_density * soil_depth)
    
    # Mean and standard deviation of soil moisture changes in m³/m³
    mean_change_m3m3 = soil_moisture_changes_m3m3.mean().item()
    std_change_m3m3 = soil_moisture_changes_m3m3.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_m3m3 / baseline_m3m3) * 100
    pct_change_std = (std_change_m3m3 / baseline_m3m3) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_m3m3": mean_change_m3m3,
        "Std_Change_m3m3": std_change_m3m3,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std,
        "Baseline_m3m3": baseline_m3m3
    })

# --- Print results ---
print("Soil Moisture Changes: 2021-2050 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(m³/m³)':>12}{'':>10}{'(m³/m³)':>12}{'':>10}")
print("-" * 80)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_m3m3']:>12.6f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_m3m3']:>12.6f}{res['Pct_Change_Std']:>10.2f}")

print(f"Mean %: Percentage of mean soil moisture change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual soil moisture changes")
print(f"Std %: Percentage of standard deviation relative to baseline")
print(f"Note: Assumes 10 cm soil depth and water density of 1000 kg/m³")
print(f"Conversion formula: (kg/m²) / (1000 kg/m³ × 0.1 m) = m³/m³")

Soil Moisture Changes: 2021-2050 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                         (m³/m³)               (m³/m³)          
--------------------------------------------------------------------------------
Global                 -0.002516     -3.11    0.000594      0.73
Tropics                -0.000094     -0.13    0.001116      1.52
Subtropics_N           -0.000686     -0.75    0.001826      1.99
Subtropics_S           -0.001509     -3.25    0.001657      3.57
Mid_Latitudes_N        -0.013078     -7.12    0.001786      0.97
Mid_Latitudes_S        -0.000115     -1.01    0.000202      1.77
Mean %: Percentage of mean soil moisture change relative to baseline
Std Change: Standard deviation of the 30 annual soil moisture changes
Std %: Percentage of standard deviation relative to baseline
Note: Assumes 10 cm soil depth and water density of 1000 kg/m³
Conversion formula: (kg/m²) / (1000 kg/m³ × 0.1 m) = m³/m³


# Runoff

In [211]:
import xarray as xr
import numpy as np
import pandas as pd

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Annual extraction ---
def extract_annual(ds):
    return ds.groupby("time.year").mean(dim="time", skipna=True)

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_base_19712000_annual.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_future585_20712100_annual.nc")

# --- Extract annual means ---
base_annual = extract_annual(ds_baseline).sel(year=slice(1971, 2000))
present_annual = extract_annual(ds_present).sel(year=slice(2071, 2100))

# --- Storage ---
results = []

# --- Regional analysis ---
for region, bounds in regions.items():
    base_reg = base_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    present_reg = present_annual.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
    
    weights = cosine_lat_weights(base_reg["lat"])
    weights_2d = weights.broadcast_like(base_reg["mrro"].isel(year=0))
    
    # Calculate annual means for baseline period (1971-2000)
    base_annual_means = (base_reg["mrro"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate annual means for present period (1981-2010) 
    present_annual_means = (present_reg["mrro"] * weights_2d).sum(dim=["lat", "lon"]) / weights_2d.sum()
    
    # Calculate baseline climatology (1971-2000 mean)
    baseline_climatology = base_annual_means.mean()
    
    # Calculate runoff changes for each year in 1981-2010 relative to baseline
    mrro_changes = present_annual_means - baseline_climatology
    
    # Convert from kg/m²/s to mm/day
    baseline_mmday = baseline_climatology.item() * 86400  # 1 kg/m²/s = 86400 mm/day
    runoff_changes_mmday = mrro_changes * 86400  # 1 kg/m²/s = 86400 mm/day
    
    # Mean and standard deviation of runoff changes in mm/day
    mean_change_mmday = runoff_changes_mmday.mean().item()
    std_change_mmday = runoff_changes_mmday.std(ddof=1).item()
    
    # Calculate percentage changes
    pct_change_mean = (mean_change_mmday / baseline_mmday) * 100
    pct_change_std = (std_change_mmday / baseline_mmday) * 100
    
    results.append({
        "Region": region,
        "Mean_Change_mmday": mean_change_mmday,
        "Std_Change_mmday": std_change_mmday,
        "Pct_Change_Mean": pct_change_mean,
        "Pct_Change_Std": pct_change_std,
        "Baseline_mmday": baseline_mmday
    })

# --- Print results ---
print("Runoff Changes: 1981-2010 relative to 1971-2000")
print(f"{'Region':<20}{'Mean Change':>12}{'Mean %':>10}{'Std Change':>12}{'Std %':>10}")
print(f"{'':>20}{'(mm/day)':>12}{'':>10}{'(mm/day)':>12}{'':>10}")
print("-" * 80)
for res in results:
    print(f"{res['Region']:<20}{res['Mean_Change_mmday']:>12.4f}{res['Pct_Change_Mean']:>10.2f}"
          f"{res['Std_Change_mmday']:>12.4f}{res['Pct_Change_Std']:>10.2f}")

print(f"\nMean Change: Average runoff change over 30 years (1981-2010)")
print(f"Mean %: Percentage of mean runoff change relative to baseline")
print(f"Std Change: Standard deviation of the 30 annual runoff changes")
print(f"Std %: Percentage of standard deviation relative to baseline")
print(f"Note: mrro represents total runoff (surface + subsurface)")
print(f"Conversion: 1 kg/m²/s = 86400 mm/day")

Runoff Changes: 1981-2010 relative to 1971-2000
Region               Mean Change    Mean %  Std Change     Std %
                        (mm/day)              (mm/day)          
--------------------------------------------------------------------------------
Global                    0.0328     11.27      0.0203      6.99
Tropics                   0.0541     13.67      0.0442     11.18
Subtropics_N              0.0433     14.80      0.0243      8.30
Subtropics_S              0.0173     32.92      0.0108     20.58
Mid_Latitudes_N           0.0186      4.99      0.0116      3.12
Mid_Latitudes_S          -0.0088    -15.69      0.0063     11.12

Mean Change: Average runoff change over 30 years (1981-2010)
Mean %: Percentage of mean runoff change relative to baseline
Std Change: Standard deviation of the 30 annual runoff changes
Std %: Percentage of standard deviation relative to baseline
Note: mrro represents total runoff (surface + subsurface)
Conversion: 1 kg/m²/s = 86400 mm/day
